# Stage 3 - Lyrics QLoRA fine-tuning (Kaggle GPU)

Fine-tunes a small instruct model (default Qwen2.5-1.5B-Instruct) in 4-bit QLoRA on your `.txt` lyrics.

**Before running:** Settings -> Accelerator -> **GPU T4 x2** (bitsandbytes 4-bit needs a GPU).

In [ ]:
import sys, os
from pathlib import Path
REPO_DIR = '/kaggle/working/metalcore'
if not Path(REPO_DIR).exists():
    !git clone https://github.com/YOUR_USERNAME/metalcore.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
!pip install -q -r requirements-lyrics.txt

In [ ]:
LYRICS_DIR = '/kaggle/input/YOUR_LYRICS_DATASET'   # folder of .txt files (one song each)
DATA_DIR   = '/kaggle/working/lyrics_dataset'
OUTPUT_DIR = '/kaggle/working/outputs/lyrics'
assert Path(LYRICS_DIR).exists(), f'Lyrics not found: {LYRICS_DIR}'

In [ ]:
# 1) Build the instruction dataset from your .txt lyrics.
!python -m lyric_training.cli build \
    --config configs/lyrics_lora.yaml \
    --lyrics {LYRICS_DIR} \
    --output {DATA_DIR}

In [ ]:
# 2) Fine-tune (resume-safe). Re-run after a session timeout with the same cell.
!python -m lyric_training.cli train \
    --config configs/lyrics_lora.yaml \
    --data {DATA_DIR} \
    --output {OUTPUT_DIR} \
    --resume

In [ ]:
# 3) Generate structured lyrics.
!python -m lyric_training.cli generate \
    --config configs/lyrics_lora.yaml \
    --adapter {OUTPUT_DIR}/adapter \
    --themes "addiction, hope" \
    --output {OUTPUT_DIR}/song.txt

In [ ]:
print(Path(OUTPUT_DIR, 'song.txt').read_text())